# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

if __name__ == "__main__":
    mcp.run()

## Exercise 1 (provide answer)

STDIO transport is simpler for local development because it doesn't require managing network ports, handling HTTP headers/authentication, or setting up a web server. It uses the standard input/output of the process, which is easier to debug and orchestrate locally.

## Exercise 2

In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # Command to run the server script using python
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            print("Connected!")

In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [ ]:
async def ex3_list():
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # List resources
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            # List tools
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
def stub_plan(prompt, functions):
    # Simple stub planner for Exercise 5
    if "Add" in prompt:
        return [{"name": "add", "args": {"a": 2, "b": 20}}]
    return []

async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # 1. Get tools
            tools_resp = await session.list_tools()
            # 2. Convert to LLM format
            functions = [convert_to_llm_tool(t) for t in tools_resp.tools]
            # 3. Get calls
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            # 4. Execute
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


# Demonstration of the new multiply tool detection
print("Re-running discovery and execution with the multiply tool...")
await ex3_list()

# Update stub_plan locally to handle multiplication for testing
def stub_plan_extended(prompt, functions):
    if "multiply" in prompt.lower():
        return [{"name": "multiply", "args": {"a": 5, "b": 4}}]
    return stub_plan(prompt, functions)

# Patch the global call_llm or just use a custom prompt
async def demo_optional():
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            print("\nTesting multiply tool:")
            result = await session.call_tool("multiply", arguments={"a": 5, "b": 4})
            print("Result of 5 * 4:", [getattr(c, "text", str(c)) for c in result.content])

await demo_optional()